In [ ]:
using Pkg
using SpeedyWeather, CairoMakie, GLMakie, Statistics

In [8]:
include("TrenberthCallbacks.jl")  # loads module into Main
using .TrenberthCallbacks

In [9]:
# spectral_grid = SpectralGrid(trunc=31, nlayers=8)
# model = PrimitiveWetModel(spectral_grid)

In [10]:
# using SpeedyWeather.Radiation
# using SpeedyWeather
# subtypes(AbstractShortwave)

spectral_grid = SpectralGrid(trunc=31, nlayers=8)
# model = PrimitiveWetModel(spectral_grid; shortwave_radiation=OneBandShortwave(spectral_grid))
model = PrimitiveWetModel(spectral_grid; shortwave_radiation=OneBandShortwave(spectral_grid, clouds = DiagnosticClouds(spectral_grid; use_stratocumulus=true)))


# # get surface shortwave radiation down
# ssrd = simulation.diagnostic_variables.physics.surface_shortwave_down
# heatmap(ssrd,title="Surface shortwave radiation down [W/m^2]")

PrimitiveWetModel <: PrimitiveWet
├ spectral_grid: SpectralGrid{CPU{KernelAbstractions.CPU}, Spectrum{CPU{KernelAbstractions.CPU}...
├ architecture: CPU{KernelAbstractions.CPU}
├ dynamics: Bool
├ geometry: Geometry{SpectralGrid{CPU{KernelAbstractions.CPU}, Spectrum{CPU{KernelAbstractions....
├ planet: Earth{Float32}
├ atmosphere: EarthAtmosphere{Float32}
├ coriolis: Coriolis{Vector{Float32}}
├ geopotential: Geopotential{Vector{Float32}}
├ adiabatic_conversion: AdiabaticConversion{Vector{Float32}}
├ particle_advection: Nothing
├ initial_conditions: InitialConditions{ZonalWind{Float32}, PressureOnOrography, JablonowskiTem...
├ forcing: Nothing
├ drag: Nothing
├ random_process: Nothing
├ tracers: Dict{Symbol, Tracer}
├ orography: EarthOrography{Float32, Field{Float32, 1, Vector{Float32}, OctahedralGaussianGrid{...
├ land_sea_mask: EarthLandSeaMask{Float32, Field{Float32, 1, Vector{Float32}, OctahedralGaussia...
├ ocean: SlabOcean{Float32}
├ sea_ice: ThermodynamicSeaIce{Float32}
├ land: La

In [11]:
simulation = initialize!(model)

Simulation{PrimitiveWetModel}
├ prognostic_variables::PrognosticVariables{...}
├ diagnostic_variables::DiagnosticVariables{...}
└ model::PrimitiveWetModel{...}

In [12]:
using SpeedyWeatherInternals
run!(simulation, period=Week(1))

LoadError: UndefVarError: `zero_last_degree!` not defined in `SpeedyWeather`
Suggestion: check for spelling errors or missing imports.

In [ ]:
# spectral_grid = SpectralGrid()

# deciding between the cloud schemes:
# use without stratocumulus clouds
# sw_no_sc = OneBandShortwave(spectral_grid, clouds = DiagnosticClouds(spectral_grid; use_stratocumulus=false))

# use with stratocumulus clouds
# sw_with_sc = OneBandShortwave(spectral_grid, clouds = DiagnosticClouds(spectral_grid; use_stratocumulus=true))

## Output variables

In [ ]:
# see the model output structure:
model.output

In [ ]:
add!(model, SpeedyWeather.RadiationOutput()...) # add radiation diagnostics to model output
SpeedyWeather.RadiationOutput()


In [ ]:
add!(model, SpeedyWeather.SurfaceFluxesOutput()...) # add surface flux diagnostics to model output

#### adding callbacks: 

In [ ]:
# --------------------------
# helper: compute Trenberth diagnostics from diagn + model
# --------------------------
function calc_trenberth_from_diagn(diagn, model; SumFlag::Bool=false) # default to global means (W/m^2), calculate from the model diagnostic variables
    fields = Dict(
        :LHF   => diagn.physics.surface_latent_heat_flux,
        :SHF   => diagn.physics.sensible_heat_flux,
        :SSRU  => diagn.physics.surface_shortwave_up,
        :SLRU  => diagn.physics.surface_longwave_up,
        :SSRD  => diagn.physics.surface_shortwave_down,
        :SLRD  => diagn.physics.surface_longwave_down,
        :OSR   => diagn.physics.outgoing_shortwave_radiation,
        :OLR   => diagn.physics.outgoing_longwave_radiation,
        :albedo=> diagn.physics.albedo
    )

    # spectral helpers (ℓ=0 → global mean; multiply by area for total)
    function calc_global_mean(field)
        a = transform(field)              # model transform -> spectral coeffs
        a00 = real(a[1])                  # index 1 == ℓ=0,m=0 (SpeedyWeather layout)
        return a00 / model.spectral_transform.norm_sphere
    end
    
    function calc_global_sum(field)
        mean_val = calc_global_mean(field)
        area = 4π * model.planet.radius^2
        return mean_val * area
    end

    calcfun = SumFlag ? calc_global_sum : calc_global_mean # pick calculation function

    results = Dict{Symbol, Float64}()
    for (k, f) in fields
        try
            results[k] = Float64(calcfun(f))
        catch err
            @warn "calc_trenberth_from_diagn: could not compute $k: $err"
            results[k] = NaN
        end
    end

    # derived surface/Trenberth terms (adjust sign convention as needed)
    results[:SW_net_sfc]  = results[:SSRD] - results[:SSRU] # calculated as down - up
    results[:LW_net_sfc]  = results[:SLRD] - results[:SLRU] # calculated as down - up
    results[:surface_net] = results[:SW_net_sfc] + results[:LW_net_sfc] - results[:LHF] - results[:SHF] # derived surface net radiation.

    return results
end


In [ ]:
# Default long names for Trenberth variables
const TRENBERTH_LONGNAMES = Dict(
    :LHF => "Surface latent heat flux (W/m²)",
    :SHF => "Surface sensible heat flux (W/m²)",
    :SSRU => "Surface shortwave up (W/m²)",
    :SLRU => "Surface longwave up (W/m²)",
    :SSRD => "Surface shortwave down (W/m²)",
    :SLRD => "Surface longwave down (W/m²)",
    :OSR => "Outgoing shortwave radiation (TOA) (W/m²)",
    :OLR => "Outgoing longwave radiation (TOA) (W/m²)",
    :albedo => "Surface albedo",
    :SW_net_sfc => "Surface net shortwave (W/m²)",
    :LW_net_sfc => "Surface net longwave (W/m²)",
    :surface_net => "Surface net energy (W/m²)"
)

In [ ]:
using Dates

# Convert various time types to Float64. Default unit = :seconds.
function time_to_float(t; unit::Symbol = :seconds)
    if t isa DateTime
        secs = Dates.datetime2unix(t)                     # seconds since Unix epoch
        return unit == :seconds ? Float64(secs) :
               unit == :days    ? Float64(secs / 86400.0) :
               error("unsupported unit: $unit")
    elseif t <: Dates.Period   # Day, Hour, Minute, etc.
        # Dates.value returns the integer magnitude in the Period's base units
        # For Day it returns number of days, for Hour number of hours, etc.
        # Convert to days or seconds depending on unit
        if unit == :days
            return float(Dates.value(t))
        elseif unit == :seconds
            # approximate: convert days/hours etc. to seconds using common ratios
            # We'll convert via Day/Hr/Minute explicitly for safety:
            if t isa Day
                return float(Dates.value(t) * 86400)
            elseif t isa Hour
                return float(Dates.value(t) * 3600)
            elseif t isa Minute
                return float(Dates.value(t) * 60)
            else
                # fallback: convert to days then seconds
                return float(Dates.value(Day(round(Int, Dates.value(t)))) * 86400)
            end
        else
            error("unsupported unit: $unit")
        end
    elseif t isa Number
        return float(t)
    else
        error("unsupported time type: $(typeof(t))")
    end
end


#### setting the callback and its schedule:

In [ ]:
Base.@kwdef mutable struct TrenberthCallback <: SpeedyWeather.AbstractCallback
    timestep_counter::Int = 0
    data::Dict{Symbol, Vector{Float64}} = Dict{Symbol, Vector{Float64}}()
    times::Vector{Float64} = Float64[]          # elapsed seconds
    datetimes::Vector{DateTime} = DateTime[]    # original DateTime stamps
    start_time::Float64 = 0.0
    SumFlag::Bool = false
    var_longnames::Dict{Symbol,String} = TRENBERTH_LONGNAMES
    schedule::Schedule = Schedule()  # default: runs every timestep
end

# Constructor function to create instances with smart allocation
function TrenberthCallback(; vars = [:LHF,:SHF,:SSRU,:SLRU,:SSRD,:SLRD,:OSR,:OLR,:albedo,:SW_net_sfc,:LW_net_sfc,:surface_net],
                             SumFlag::Bool=false,
                             nsteps::Int=0,
                             var_longnames::Dict{Symbol,String}=TRENBERTH_LONGNAMES,
                             schedule::Schedule=Schedule())
    d = Dict{Symbol, Vector{Float64}}()
    for v in vars
        d[v] = nsteps > 0 ? Vector{Float64}(undef, nsteps + 1) : Float64[]
    end
    times = nsteps > 0 ? Vector{Float64}(undef, nsteps + 1) : Float64[]
    datetimes = nsteps > 0 ? Vector{DateTime}(undef, nsteps + 1) : DateTime[]
    return TrenberthCallback(0, d, times, datetimes, 0.0, SumFlag, var_longnames, schedule)
end


In [ ]:
# Run every timestep (default)
# cb = TrenberthCallback(SumFlag=false, nsteps=0)

# Run every day
# cb = TrenberthCallback(SumFlag=false, nsteps=0, schedule=Schedule(every=Day(1)))

# Run every 6 hours
# cb = TrenberthCallback(SumFlag=false, nsteps=0, schedule=Schedule(every=Hour(6)))

# Run every 12 hours
cb = TrenberthCallback(SumFlag=false, nsteps=0, schedule=Schedule(every=Hour(12)))

In [ ]:
# Pretty-print the long names
function show_var_names(cb::TrenberthCallback)
    for (k, long) in cb.var_longnames
        println(string(k), " → ", long)
    end
    return nothing
end

# assemble a DataFrame if DataFrames.jl is installed
function to_dataframe(cb::TrenberthCallback)
    try
        @eval using DataFrames
    catch
        error("DataFrames.jl not available. Install it with `using Pkg; Pkg.add(\"DataFrames\")`")
    end
    df = DataFrame(time = cb.datetimes)
    for (k, vec) in cb.data
        colname = get(cb.var_longnames, k, string(k))  # column name uses long name if available
        # ensure column identifier is a Symbol
        df[Symbol(colname)] = vec
    end
    return df
end

In [ ]:
function SpeedyWeather.initialize!(cb::TrenberthCallback,
                                   progn::PrognosticVariables,
                                   diagn::DiagnosticVariables,
                                   model::AbstractModel)
    
    # when initializing a scheduled callback also initialize its schedule!
    initialize!(cb.schedule, progn.clock)

    # Store the simulation start time for reference (convert DateTime to Float64 Unix timestamp)
    cb.start_time = Dates.datetime2unix(progn.clock.time)
    
    # Try to get nsteps, but if it doesn't work, just start with empty vectors
    try
        nsteps = progn.clock.nsteps
        # if our data dict vectors are empty or wrong size, (re)allocate
        for (k, v) in cb.data
            if isempty(v) || length(v) != nsteps + 1
                cb.data[k] = Vector{Float64}(undef, nsteps + 1)
            end
        end
        if isempty(cb.times) || length(cb.times) != nsteps + 1
            cb.times = Vector{Float64}(undef, nsteps + 1)
        end
        if isempty(cb.datetimes) || length(cb.datetimes) != nsteps + 1
            cb.datetimes = Vector{DateTime}(undef, nsteps + 1)
        end
    catch
        # If we can't get nsteps, just use dynamic push mode
        @info "Could not determine nsteps, using dynamic push mode"
    end

    # set counter to 1 and store initial conditions
    cb.timestep_counter = 1
    t0 = Dates.datetime2unix(progn.clock.time)  # Convert DateTime to Unix timestamp
    dt0 = progn.clock.time  # Get the original DateTime object
    # compute initial values using diagn
    res0 = calc_trenberth_from_diagn(diagn, model; SumFlag=cb.SumFlag)
    for (k, v) in res0
        if haskey(cb.data, k) # check if key exists
            if length(cb.data[k]) > 0
                cb.data[k][1] = v
            else
                push!(cb.data[k], v)
            end
        else
            cb.data[k] = [v]
        end
    end

    # Store time relative to simulation start (in seconds) and DateTime
    if length(cb.times) > 0
        cb.times[1] = t0 - cb.start_time
        cb.datetimes[1] = dt0
    else
        push!(cb.times, t0 - cb.start_time)
        push!(cb.datetimes, dt0)
    end
    return nothing
end

In [ ]:
# --- callback! called every step (after the step completes) ---
function SpeedyWeather.callback!(cb::TrenberthCallback,
                                 progn::PrognosticVariables,
                                 diagn::DiagnosticVariables,
                                 model::AbstractModel)
    
    # scheduled callbacks start with this line to execute only when scheduled!
    # else escape immediately
    isscheduled(cb.schedule, progn.clock) || return nothing

    # increment step index
    cb.timestep_counter += 1
    
    # compute current diagnostics
    res = calc_trenberth_from_diagn(diagn, model; SumFlag=cb.SumFlag)
    
    # push new values to the arrays
    for (k, v) in res
        if !haskey(cb.data, k)
            # new key appeared: create vector and push
            cb.data[k] = [v]
        else
            # existing key: push to the vector
            push!(cb.data[k], v)
        end
    end
    
    # record model time relative to simulation start (in seconds) and DateTime
    # Convert DateTime to Unix timestamp, then subtract start_time to get elapsed seconds
    current_time = Dates.datetime2unix(progn.clock.time)
    push!(cb.times, current_time - cb.start_time)
    push!(cb.datetimes, progn.clock.time)  # Store the DateTime object
    return nothing
end

In [ ]:
# --- finalize (optional) ---
using Statistics  # Import mean function

# use the finalize! to clculate the mean values over the entire simulation:
function SpeedyWeather.finalize!(cb::TrenberthCallback,
                                   progn::PrognosticVariables,
                                   diagn::DiagnosticVariables,
                                   model::AbstractModel)
    # compute final diagnostics
    # res = calc_trenberth_from_diagn(diagn, model; SumFlag=cb.SumFlag)
    # for (k, v) in res
    #     if haskey(cb.data, k)
    #         push!(cb.data[k], v)
    #     else
    #         cb.data[k] = [v]
    #     end
    # end

    # compute the mean over the entire simulation for each variable and print
    # Note: we keep the original data vectors and just print the means
    println("\n=== Simulation Means ===")
    for (k, vec) in cb.data
        mean_val = mean(vec)
        println("Mean $k over simulation: $mean_val")
    end
    return nothing
end

adding the calbak to the model


In [ ]:
# A: add into the callbacks dict directly (works if model.callbacks is a Dict-like)
add!(model.callbacks, :trenberth => cb)

In [ ]:
keys(model.callbacks)              # should include :trenberth to make sure the callback was added correctly.

In [ ]:
model.callbacks[:trenberth] === cb # should be true to make sure the callback was added correctly. 

In [ ]:
# SpeedyWeather.readable_secs

In [ ]:
sim = initialize!(model)   # this will call SpeedyWeather.initialize! on cb
run!(sim, period=Day(10))  # or your usual run invocation

In [ ]:
cb.data # should contain the Trenberth variables collected during the simulation

In [ ]:
size(cb.data[:LHF]) # should show the number of time steps + 1 (for initial condition)

In [ ]:
cb.data[:LHF][:] # should show the last 10 values of the latent heat flux variable

In [ ]:
size(cb.times) # should show the number of time steps + 1 (for initial condition)

In [ ]:
cb.times

In [ ]:
size(cb.datetimes) # should show the number of time steps + 1 (for initial condition)

In [ ]:
cb.times

In [ ]:
cb.datetimes # to see the DateTime objects corresponding to the times

In [ ]:
cb.var_longnames